# 05 - Evaluation generations + pairwise files (Task 6)
**Accelerator:** GPU T4 x2 · **Internet:** ON · **Inputs:** outputs of notebooks 01, 02, 03a, 03b · **Runtime:** ~30-60 min

Greedy decoding, identical settings for every model in a comparison (see `*.meta.json`).

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"   # live script output in the Kaggle log
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
# Keep Kaggle's CUDA-matched PyTorch. Do NOT reinstall torch.
sh(f"""python -m pip uninstall -y -q torchao || true""")
sh(f"""python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest""")
sh(f"""python -m pip install -q -e . --no-deps""")

In [ ]:
# Bring the fixed data files from notebook 01's output into the repo
os.makedirs("data/processed", exist_ok=True)
src = os.path.dirname(find_input("data/processed/manifest.json"))
for name in os.listdir(src):
    shutil.copy(os.path.join(src, name), "data/processed/")
shutil.copy(find_input("eval_sets/alpaca_eval_300_seed42.json"), "eval_sets/")
print(sorted(os.listdir("data/processed")))

In [ ]:
CKPT = {
    'sft': os.path.dirname(find_input('sft/best/model.safetensors')),
    'dpo_hh': os.path.dirname(find_input('dpo_hh/best/model.safetensors')),
    'dpo_uf': os.path.dirname(find_input('dpo_uf/best/model.safetensors')),
}
print(CKPT)
os.makedirs('outputs/generations', exist_ok=True)
for name in ('base_alpaca.json', 'base_alpaca.meta.json', 'base_hh.json', 'base_hh.meta.json'):
    try: shutil.copy(find_input(f'outputs/generations/{name}'), 'outputs/generations/')
    except FileNotFoundError as e: print(e)

## Sanity check (2-5 prompts per checkpoint): path, tokenizer, prompt format, EOS behaviour, length

In [ ]:
for key, path in CKPT.items():
    sh(f"""python scripts/generate.py --model_path {path} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator sanity-{key} --output outputs/generations/sanity_{key}.json --limit 3 --print_samples --overwrite""")

## Full generations
SFT is generated on both sets (it is the baseline for both comparisons). Each DPO model on its own eval set; the cross-set generations are optional extras for error analysis.

In [ ]:
JOBS = [
    ('sft',    'alpaca', 'qwen2.5-0.5b-sft'),
    ('sft',    'hh',     'qwen2.5-0.5b-sft'),
    ('dpo_uf', 'alpaca', 'qwen2.5-0.5b-dpo-uf'),
    ('dpo_hh', 'hh',     'qwen2.5-0.5b-dpo-hh'),
    # optional cross-set analysis (alignment tax / over-refusal):
    ('dpo_hh', 'alpaca', 'qwen2.5-0.5b-dpo-hh'),
    ('dpo_uf', 'hh',     'qwen2.5-0.5b-dpo-uf'),
]
EVAL = {'alpaca': 'eval_sets/alpaca_eval_300_seed42.json', 'hh': 'eval_sets/hh_rlhf_eval_300_seed42.json'}
for key, eval_name, generator in JOBS:
    out = f'outputs/generations/{key}_{eval_name}.json'
    sh(f"""python scripts/generate.py --model_path {CKPT[key]} --eval_set {EVAL[eval_name]} --generator {generator} --output {out}""")

## Blinded pairwise files (DPO = candidate, SFT = baseline) + an A/B-swapped subset for position-bias checks

In [ ]:
sh(f"""python scripts/build_pairs.py --candidate outputs/generations/dpo_uf_alpaca.json --baseline outputs/generations/sft_alpaca.json --output outputs/eval/dpo_uf_vs_sft_pairs.jsonl""")
sh(f"""python scripts/build_pairs.py --candidate outputs/generations/dpo_hh_hh.json --baseline outputs/generations/sft_hh.json --output outputs/eval/dpo_hh_vs_sft_pairs.jsonl""")
sh(f"""python scripts/build_pairs.py --mirror_of outputs/eval/dpo_uf_vs_sft_pairs.jsonl --limit 100 --output outputs/eval/dpo_uf_vs_sft_mirrored_pairs.jsonl""")
sh(f"""python scripts/build_pairs.py --mirror_of outputs/eval/dpo_hh_vs_sft_pairs.jsonl --limit 100 --output outputs/eval/dpo_hh_vs_sft_mirrored_pairs.jsonl""")

In [ ]:
sh(f"""python scripts/analyze_generations.py outputs/generations/*.json --output outputs/eval/generation_stats.json""")

In [ ]:
# Lightweight archive (metrics/logs/plots/generations - not the 2 GB checkpoints)
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/05_generations.zip outputs/generations outputs/eval eval_sets -x "*/best/*" "*/final/*" "*/trainer/*" && ls -lh /kaggle/working/05_generations.zip""")
sh(f"""df -h /kaggle/working | tail -1""")